## Setup Colab and sync


In [1]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 1800, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 1800 (delta 27), reused 41 (delta 20), pack-reused 1742 (from 2)
Receiving objects: 100% (1800/1800), 197.70 MiB | 16.30 MiB/s, done.
Resolving deltas: 100% (986/986), done.


In [2]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [3]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

python3: can't open file '/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py': [Errno 2] No such file or directory


In [4]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Set precision

In [5]:
# Example: copy MyDrive/data_big to local /content/data_big
!mkdir -p "/content/emb"
!rsync -a --info=progress2 \
  --include='all_embeddings_float16.pkl' \
  --exclude='*' \
  "/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/" \
  "/content/emb/"


 18,990,563,947 100%   50.50MB/s    0:05:58 (xfr#1, to-chk=0/2)


## Load BERT embeddings of all patents

In [6]:
import os
import glob
import pickle
import numpy as np

# Define the output file path for the combined embeddings.
filepath = '/content/emb/all_embeddings_float16.pkl'

# Load
with open(filepath, 'rb') as f:
    all_embeddings_dict = pickle.load(f)


In [7]:
print(len(all_embeddings_dict))

9075421


In [10]:

all_embeddings_dict['6347138']

array([-1.13  , -2.158 , -1.233 , ..., -0.901 ,  0.713 ,  0.9526],
      dtype=float16)

In [8]:
all_embeddings_dict['4659751']

array([-0.08453,  0.7515 , -0.41   , ..., -0.9077 ,  0.861  , -0.4875 ],
      dtype=float16)

In [ ]:
from sklearn.decomposition import IncrementalPCA
import numpy as np
import pandas as pd
import os

# Assume all_embeddings_dict = {patent_id: np.array([...])}
batch_size = 250000
n_components_list = [15, 20, 25, 30]
output_folder = "/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/PCA/"
os.makedirs(output_folder, exist_ok=True)

# Create IncrementalPCA models for 3–10 PCs
pca_models = {k: IncrementalPCA(n_components=k, batch_size=batch_size) for k in n_components_list}

# Convert dict to iterable of (key, vector)
items = list(all_embeddings_dict.items())
n = len(items)

# === FIRST PASS: partial_fit each PCA ===
for i in range(0, n, batch_size):
    batch = items[i:i+batch_size]
    X = np.vstack([v for _, v in batch])
    for pca in pca_models.values():
        pca.partial_fit(X)
    print(f"Fitted batch {i//batch_size+1}/{n//batch_size+1}")

# === SECOND PASS: transform and save ===
for k, pca in pca_models.items():
    output_path = os.path.join(output_folder, f"pca_{k}D.csv")
    print(f"\n--- Transforming and saving PCA ({k} components) ---")

    # Write header first
    with open(output_path, "w") as f:
        header = ["patent_id"] + [f"PC{i+1}" for i in range(k)]
        f.write(",".join(header) + "\n")

    # Stream transformation batch by batch
    for i in range(0, n, batch_size):
        batch = items[i:i+batch_size]
        ids_batch, X = zip(*batch)
        X = np.vstack(X)
        X_pca = pca.transform(X)

        # Create a small DataFrame for just this batch
        df_batch = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(k)])
        df_batch.insert(0, "patent_id", ids_batch)

        # Append to CSV (no header)
        df_batch.to_csv(output_path, mode="a", header=False, index=False)

        print(f"  Wrote batch {i//batch_size+1}/{(n-1)//batch_size+1}")

    print(f"✅ Saved PCA ({k}D) to {output_path} | variance explained: {pca.explained_variance_ratio_.sum():.2%}")


Fitted batch 1/37
Fitted batch 2/37
Fitted batch 3/37
Fitted batch 4/37
Fitted batch 5/37
Fitted batch 6/37
Fitted batch 7/37
Fitted batch 8/37
Fitted batch 9/37
Fitted batch 10/37
Fitted batch 11/37
Fitted batch 12/37
Fitted batch 13/37
Fitted batch 14/37
Fitted batch 15/37
Fitted batch 16/37
Fitted batch 17/37
Fitted batch 18/37
Fitted batch 19/37
Fitted batch 20/37
Fitted batch 21/37
Fitted batch 22/37
Fitted batch 23/37
Fitted batch 24/37
Fitted batch 25/37
Fitted batch 26/37
Fitted batch 27/37
Fitted batch 28/37
Fitted batch 29/37
Fitted batch 30/37
Fitted batch 31/37
Fitted batch 32/37
Fitted batch 33/37
Fitted batch 34/37
Fitted batch 35/37
Fitted batch 36/37
Fitted batch 37/37

--- Transforming and saving PCA (15 components) ---
  Wrote batch 1/37
  Wrote batch 2/37
  Wrote batch 3/37
  Wrote batch 4/37
  Wrote batch 5/37
  Wrote batch 6/37
  Wrote batch 7/37
  Wrote batch 8/37
  Wrote batch 9/37
  Wrote batch 10/37
  Wrote batch 11/37
  Wrote batch 12/37
  Wrote batch 13/37
  